In [2]:
%pip install pylatexenc
%pip install -U langchain-google-vertexai
%pip install -U langchain-community
%pip install -U langchain-huggingface
#%pip install faiss-gpu
%pip install faiss-cpu
%pip install regex
%pip install chardet
%pip install pytictoc

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode
from pylatexenc.latex2text import LatexNodes2Text

In [4]:
import tarfile
import zipfile
import io
import os
import time
import itertools as itr
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import requests

# Phase 2
import json
import vertexai
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import storage

from langchain.docstore.document import Document
from langchain_google_vertexai import VertexAI
from langchain.vectorstores import FAISS
#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

PROJECT_ID = "arxiv-development"
vertexai.init(project=PROJECT_ID, location="us-central1")

In [5]:
!curl -o phase_one_ex.py https://raw.githubusercontent.com/arXiv/metadata-vertexai/refs/heads/cjc73-working/phase_one.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 24787  100 24787    0     0   175k      0 --:--:-- --:--:-- --:--:--  175k


In [3]:
#os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_ex as phase_one

Note that id lists were prepared previously from the DB, using:  

```
select concat(paper_id,"v",version) as arx_id from arXiv_metadata
where paper_id LIKE "23%";
and is_withdrawn != 1
and is_current = 1;
```

In [5]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
test_ids_df.head()

,arx_id
0,2311.00001v1
1,2311.00002v1
2,2311.00003v4
3,2311.00004v3
4,2311.00005v1


### test phase 1

In [6]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip")
scopus_all = scopus_df["ArXiv Id"].unique()
scopus_positive = scopus_df[scopus_df['Primary Org Id'] == 60027550]["ArXiv Id"].unique()
false_positive = [
  "2311.00030v1",
  "2311.00088",
  "2311.00094",
  "2311.00145",
  "2311.00770",
  "2311.02280",
  "2311.02468",
  "2311.03261",
  "2311.03309",
  "2311.03527v2",
  "2311.04856",
  "2311.04862",
  "2311.04942",
  "2311.05674",
  "2311.05678",
  "2311.08133",
  "2311.09521",
  "2311.09562",
  "2311.09638",
  "2311.09734",
  "2311.10140",
  "2311.10604",
  "2311.10985",
  "2311.12103",
  "2311.12152",
  "2311.12263",
  "2311.12334",
  "2311.12656",
  "2311.13107",
  "2311.13244",
  "2311.13718",
  "2311.13900",
  "2311.13907",
  "2311.14103",
  "2311.14409",
  "2311.14599",
  "2311.15441",
  "2311.15533",
  "2311.15541",
  "2311.16400",
  "2311.17162",
  "2311.17252",
  "2311.17314",
  "2311.17844",
  "2311.17915"
]

### Threaded processing for multiple files

## Test

## times

```
Batch size    parallel workers    thread workers    time               n       sec/item        errors
   5             2                   5                                 100     
  10             2                   5                                 100     
  10             2                  10                                 100     
  
   5             3                   5                2 min 50s        100     1.70
  10             3                   5                2 min  8s        100     1.28
  10             3                  10                2 min 23s        100     1.43
  
  10             8                  10                3min 44s        1000      .22              0
  20             8                  10                3min 15s.       1000      .21              0       6414, 255; 6717, 296
  20             8                  20                3min 27s        1000      .21              0
  50             8                  25                3min 21s        1000      .20 
  
  25            12                  25                8min 1s         1000      .48
 100            12                  25                12min 39s       1000      .73 
 

10                   3                          5                    1 min 14s
```

In [18]:
%%time

tt = TicToc()

sample_size = 1000
batch_size = 20
parallel_workers = 8
thread_workers = 10
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def format_results(arxid_inst_ror_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_ror_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = 'null'
            if len(x) == 3:
                ror = x[2].strip()
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list

os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()

res_df = pd.DataFrame.from_records(res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/2311_scopus_{sample_size}.csv.zip", index=False)

tt.toc()

Start Phase 1, 1000 in 50 batches
✅ Total processing time: 30.32 seconds
✅ Total processing time: 31.76 seconds
✅ Total processing time: 37.48 seconds
✅ Total processing time: 19.56 seconds
✅ Total processing time: 21.64 seconds
✅ Total processing time: 57.29 seconds
✅ Total processing time: 62.23 seconds
✅ Total processing time: 67.51 seconds
✅ Total processing time: 71.43 seconds
✅ Total processing time: 24.85 seconds
✅ Total processing time: 41.50 seconds
✅ Total processing time: 22.28 seconds
✅ Total processing time: 81.00 seconds
✅ Total processing time: 26.44 seconds
✅ Total processing time: 39.90 seconds
✅ Total processing time: 24.71 seconds
✅ Total processing time: 22.22 seconds
✅ Total processing time: 22.26 seconds
✅ Total processing time: 22.46 seconds
✅ Total processing time: 23.96 seconds
✅ Total processing time: 24.11 seconds
✅ Total processing time: 19.91 seconds
✅ Total processing time: 23.83 seconds
✅ Total processing time: 28.50 seconds
✅ Total processing time: 16.37

[('2311.02430', 'University of Delhi, Delhi', '04gzb2213'),
 ('2311.11008', 'Nazarbayev University,', '052bx8q98'),
 ('2311.18483', 'University of Calcutta,', '01e7v7w47'),
 ('2311.18483',
  'Indian Institute of Science Education and Research Kolkata,',
  '00djv2c17'),
 ('2311.04492', 'National Institute of Technology, Patna', '056wyhh33')]

In [19]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

6717

0

296